In [16]:
import os
import pandas as pd
import numpy as np
import librosa
from scipy.signal import convolve2d
from sklearn.svm import SVC
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

# Use tqdm.notebook for a cleaner progress bar inside Jupyter
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore') # Suppresses librosa STFT warnings for cleaner output

In [2]:
def compute_lpq_histogram(img, win_size=3):
    """
    Computes Local Phase Quantization (LPQ) texture features.
    Returns a fixed-size 256-bin histogram for the given 2D spectrogram zone.
    """
    radius = (win_size - 1) // 2
    x = np.arange(-radius, radius + 1)
    y = np.arange(-radius, radius + 1)
    [x, y] = np.meshgrid(x, y)
    
    u0, u1 = 1/win_size, 0
    v0, v1 = 0, 1/win_size
    
    f1 = np.exp(-2 * np.pi * 1j * (u0 * x + v0 * y))
    f2 = np.exp(-2 * np.pi * 1j * (u1 * x + v1 * y))
    f3 = np.exp(-2 * np.pi * 1j * (u0 * x + v1 * y))
    f4 = np.exp(-2 * np.pi * 1j * (u1 * x - v0 * y))
    
    filters = [f1.real, f1.imag, f2.real, f2.imag, 
               f3.real, f3.imag, f4.real, f4.imag]
    
    lpq_img = np.zeros(img.shape, dtype=np.uint8)
    for i, f in enumerate(filters):
        resp = convolve2d(img, f, mode='same', boundary='symm')
        lpq_img += (resp > 0).astype(np.uint8) * (2 ** i)
        
    hist, _ = np.histogram(lpq_img.flatten(), bins=256, range=(0, 256))
    return hist

def extract_features(y, sr=8000, num_zones=10):
    """
    Accepts a 1D numpy array waveform, generates a db-scaled spectrogram, 
    normalizes to an 8-bit image, splits into linear frequency zones, 
    and extracts LPQ histograms per zone.
    """
    # Ensure the input is a float32 numpy array (librosa standard)
    if not isinstance(y, np.ndarray):
        y = np.array(y, dtype=np.float32)
    
    # 1. Acoustic Front-End (n_fft=256, 10ms hop length, Hann window)
    hop_length = int(sr * 0.01)
    D = librosa.stft(y, n_fft=256, hop_length=hop_length, window='hann')
    
    # 2. Log-Scaling & Image Conversion
    S_db = librosa.amplitude_to_db(np.abs(D), ref=np.max)
    
    # Normalize linearly to [0, 255] for visual texture processing
    S_min, S_max = S_db.min(), S_db.max()
    if S_max > S_min:
        S_img = ((S_db - S_min) / (S_max - S_min) * 255).astype(np.uint8)
    else:
        S_img = np.zeros_like(S_db, dtype=np.uint8)
        
    # 3. Frequency Zoning
    freq_bins = S_img.shape[0]
    bins_per_zone = freq_bins // num_zones
    
    zone_histograms = []
    for i in range(num_zones):
        start_bin = i * bins_per_zone
        end_bin = (i + 1) * bins_per_zone if i < num_zones - 1 else freq_bins
        zone = S_img[start_bin:end_bin, :]
        
        # 4. LPQ Extraction
        hist = compute_lpq_histogram(zone)
        zone_histograms.append(hist)
        
    return np.array(zone_histograms)

def build_svm_pipeline():
    """Configures the SVM with exact paper methodology settings."""
    return Pipeline([
        ('scaler', MinMaxScaler(feature_range=(-1, 1))), 
        ('svm', SVC(kernel='rbf', probability=True))     
    ])

In [3]:
def process_dataset(df, signal_column='signal', label_column='language_encoded', sample_rate=8000):
    """Iterates through a dataframe to extract features from audio arrays."""
    all_features = []
    labels = []
    
    for index, row in tqdm(df.iterrows(), total=len(df), desc="Extracting Features"):
        try:
            # Grab the waveform array directly from the dataframe
            waveform = row[signal_column]
            
            # Pass the array to the feature extractor
            features = extract_features(waveform, sr=sample_rate, num_zones=10)
            
            all_features.append(features)
            labels.append(row[label_column])
            
        except Exception as e:
            print(f"Skipping row {index} due to error: {e}")
            
    # Stack into 3D array: (num_samples, 10_zones, 256_features)
    X_3d = np.stack(all_features)
    y = np.array(labels)
    
    return X_3d, y

# ==========================================
# TODO: Load your actual dataframes here
# train_set = pd.read_csv('your_train_data.csv')
# test_set = pd.read_csv('your_test_data.csv')
# ==========================================

In [17]:
df = pd.read_pickle('RomanceDigitData.pkl')
df.head()

,signal_data,language,speaker,label,gender,c,length
0,"[0.0, 3.0517578125e-05, -6.103515625e-05, -3.0...",PO,A,1,M,c,7101
1,"[0.0, 0.0, 3.0517578125e-05, 0.0, 6.103515625e...",PO,A,2,M,c,12861
2,"[0.0, 0.0, 0.0, -3.0517578125e-05, 0.0, 3.0517...",PO,A,3,M,c,10557
3,"[0.0, 0.0, -3.0517578125e-05, -3.0517578125e-0...",PO,A,4,M,c,10941
4,"[-3.0517578125e-05, 3.0517578125e-05, 0.0, 0.0...",PO,A,5,M,c,14397


In [18]:
from sklearn.model_selection import StratifiedShuffleSplit

# Simply split based on the language column
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

for train_index, test_index in sss.split(df, df['language']):
    train_set = df.iloc[train_index]
    test_set = df.iloc[test_index]

print(f"Lang Train distribution:\n{train_set['language'].value_counts()}")
print(f"Lang Test distribution:\n{test_set['language'].value_counts()}")

print(f"Num Train distribution:\n{train_set['label'].value_counts()}")
print(f"Num Test distribution:\n{test_set['label'].value_counts()}")

Lang Train distribution:
language
FR    48
IT    40
SA    37
SI    30
PO    20
Name: count, dtype: int64
Lang Test distribution:
language
FR    12
IT    10
SA     9
SI     8
PO     5
Name: count, dtype: int64
Num Train distribution:
label
2     21
4     20
7     19
5     18
1     18
6     17
9     16
8     16
3     15
10    15
Name: count, dtype: int64
Num Test distribution:
label
8     7
3     7
9     6
1     5
5     5
6     4
7     3
10    3
4     2
2     2
Name: count, dtype: int64


In [20]:
train_set.to_pickle("train_set.pkl")

In [21]:
test_set.to_pickle("test_set.pkl")

In [6]:
for num in sorted(train_set['label'].unique()):
    for lang in sorted(train_set['language'].unique()):
        # Filter the dataframe for the specific language AND label
        count = len(train_set[(train_set['language'] == lang) & (train_set['label'] == num)])
        print(f"Language: {lang}, Label: {num}, Count: {count}")

Language: FR, Label: 1, Count: 4
Language: IT, Label: 1, Count: 5
Language: PO, Label: 1, Count: 2
Language: SA, Label: 1, Count: 3
Language: SI, Label: 1, Count: 4
Language: FR, Label: 2, Count: 5
Language: IT, Label: 2, Count: 5
Language: PO, Label: 2, Count: 3
Language: SA, Label: 2, Count: 4
Language: SI, Label: 2, Count: 4
Language: FR, Label: 3, Count: 6
Language: IT, Label: 3, Count: 2
Language: PO, Label: 3, Count: 2
Language: SA, Label: 3, Count: 3
Language: SI, Label: 3, Count: 2
Language: FR, Label: 4, Count: 5
Language: IT, Label: 4, Count: 4
Language: PO, Label: 4, Count: 2
Language: SA, Label: 4, Count: 5
Language: SI, Label: 4, Count: 4
Language: FR, Label: 5, Count: 6
Language: IT, Label: 5, Count: 4
Language: PO, Label: 5, Count: 1
Language: SA, Label: 5, Count: 4
Language: SI, Label: 5, Count: 3
Language: FR, Label: 6, Count: 5
Language: IT, Label: 6, Count: 2
Language: PO, Label: 6, Count: 2
Language: SA, Label: 6, Count: 5
Language: SI, Label: 6, Count: 3
Language: 

In [7]:
print("Processing Training Set...")
# If your dataset was sampled at 16kHz, change sample_rate=16000 
# (Though the paper specifically uses 8kHz for the STFT calculation)
# If your df.head() shows the arrays are in a column named 'audio'
X_train_3d, y_train = process_dataset(
    train_set, 
    signal_column='signal_data', # <--- Update this to match your dataframe exactly
    label_column='language', # <--- Notice I also updated this to match your 'language' column!
    sample_rate=8000
)

X_test_3d, y_test = process_dataset(
    test_set, 
    signal_column='signal_data', # <--- Update this here too
    label_column='language', 
    sample_rate=8000
)

print("Processing Test Set...")
# X_test_3d, y_test = process_dataset(test_set, signal_column='signal_data', label_column='language_encoded', sample_rate=8000)

print(f"\nFeature extraction complete!")
print(f"Train shape: {X_train_3d.shape}")
print(f"Test shape: {X_test_3d.shape}")

Processing Training Set...


Extracting Features: 100%|██████████| 44/44 [00:00<00:00, 343.78it/s]

Processing Test Set...

Feature extraction complete!
Train shape: (175, 10, 256)
Test shape: (44, 10, 256)


In [8]:
df

,signal_data,language,speaker,label,gender,c,length
0,"[0.0, 3.0517578125e-05, -6.103515625e-05, -3.0...",PO,A,1,M,c,7101
1,"[0.0, 0.0, 3.0517578125e-05, 0.0, 6.103515625e...",PO,A,2,M,c,12861
2,"[0.0, 0.0, 0.0, -3.0517578125e-05, 0.0, 3.0517...",PO,A,3,M,c,10557
3,"[0.0, 0.0, -3.0517578125e-05, -3.0517578125e-0...",PO,A,4,M,c,10941
4,"[-3.0517578125e-05, 3.0517578125e-05, 0.0, 0.0...",PO,A,5,M,c,14397
...,...,...,...,...,...,...,...
214,"[-0.140899658203125, -0.141082763671875, -0.14...",SA,E,5,F,c,13001
215,"[-0.139862060546875, -0.1397705078125, -0.1402...",SA,E,6,F,c,13501
216,"[-0.145751953125, -0.146148681640625, -0.14587...",SA,E,7,F,c,13001
217,"[-0.138397216796875, -0.138275146484375, -0.14...",SA,E,8,F,c,8201


In [9]:
# Initialize a list to hold the 10 separate SVM pipelines
models = [build_svm_pipeline() for _ in range(10)]

print("Training SVM models for each of the 10 frequency zones...")
for i in tqdm(range(10), desc="Training SVMs"):
    # Extract the features specific to the i-th zone for all training samples
    X_zone_train = X_train_3d[:, i, :] 
    models[i].fit(X_zone_train, y_train)

print("Training complete.")

Training SVM models for each of the 10 frequency zones...


Training SVMs: 100%|██████████| 10/10 [00:00<00:00, 95.52it/s]

Training complete.


In [10]:
from sklearn.model_selection import GridSearchCV

# Initialize lists to store the best models and their optimal parameters
optimized_models = []
zone_best_params = []

print("Tuning and training SVM models for each of the 10 frequency zones...")

# The parameter dictionary keys must use 'svm__' prefix to match the name 
# of the SVC step in your build_svm_pipeline() function
param_grid = {
    'svm__C': [0.1, 1, 10, 100],
    'svm__gamma': ['scale', 'auto', 0.01, 0.1, 1]
}

for i in tqdm(range(10), desc="Tuning SVMs"):
    # Extract the features specific to the i-th zone
    X_zone_train = X_train_3d[:, i, :] 
    
    # Generate a fresh pipeline for this zone
    pipeline = build_svm_pipeline()
    
    # Set up the Grid Search (n_jobs=-1 uses all CPU cores to speed it up)
    grid_search = GridSearchCV(pipeline, param_grid, cv=3, scoring='accuracy', n_jobs=-1)
    
    # Fit the grid search to find the optimal C and gamma for THIS specific frequency band
    grid_search.fit(X_zone_train, y_train)
    
    # Save the best fitted pipeline so it can be used in the Fusion rules
    optimized_models.append(grid_search.best_estimator_)
    zone_best_params.append(grid_search.best_params_)
    
    # Print the findings for this zone
    print(f"Zone {i+1} Best Params: {grid_search.best_params_} (CV Acc: {grid_search.best_score_ * 100:.2f}%)")

print("\nHyperparameter tuning and training complete.")

Tuning and training SVM models for each of the 10 frequency zones...


Tuning SVMs:  20%|██        | 2/10 [00:01<00:06,  1.24it/s]

Zone 1 Best Params: {'svm__C': 10, 'svm__gamma': 0.01} (CV Acc: 62.85%)
Zone 2 Best Params: {'svm__C': 10, 'svm__gamma': 0.1} (CV Acc: 47.99%)


Tuning SVMs:  40%|████      | 4/10 [00:02<00:02,  2.94it/s]

Zone 3 Best Params: {'svm__C': 10, 'svm__gamma': 0.01} (CV Acc: 45.76%)
Zone 4 Best Params: {'svm__C': 10, 'svm__gamma': 'scale'} (CV Acc: 49.74%)


...

Zone 5 Best Params: {'svm__C': 10, 'svm__gamma': 0.01} (CV Acc: 49.18%)
Zone 6 Best Params: {'svm__C': 10, 'svm__gamma': 0.1} (CV Acc: 53.69%)


Tuning SVMs:  80%|████████  | 8/10 [00:02<00:00,  5.88it/s]

Zone 7 Best Params: {'svm__C': 100, 'svm__gamma': 'scale'} (CV Acc: 47.41%)
Zone 8 Best Params: {'svm__C': 100, 'svm__gamma': 'scale'} (CV Acc: 55.44%)


Tuning SVMs: 100%|██████████| 10/10 [00:02<00:00,  3.46it/s]

Zone 9 Best Params: {'svm__C': 10, 'svm__gamma': 0.01} (CV Acc: 47.46%)
Zone 10 Best Params: {'svm__C': 10, 'svm__gamma': 0.01} (CV Acc: 50.29%)

Hyperparameter tuning and training complete.


In [ ]:
'''Without Gridsearch'''
# Create an array to hold the probabilities
num_languages = len(np.unique(y_train)) 
all_test_probs = np.zeros((len(test_set), 10, num_languages))

# 1. Gather probability predictions from all 10 models
for i in range(10):
    X_zone_test = X_test_3d[:, i, :]
    all_test_probs[:, i, :] = models[i].predict_proba(X_zone_test)
    # all_test_probs[:, i, :] = optimized_models[i].predict_proba(X_zone_test)

# Fetch the exact string labels the SVM learned (e.g., ['FR', 'IT', 'PO', 'SA', 'SI'])
class_labels = models[0].classes_

# 2. PRODUCT RULE FUSION
prod_test_probs = np.prod(all_test_probs, axis=1)
# argmax gets the integer index (0-4), class_labels maps it back to the string name
final_prod_predictions = class_labels[np.argmax(prod_test_probs, axis=1)] 
prod_accuracy = accuracy_score(y_test, final_prod_predictions)

# 3. SUM RULE FUSION
sum_test_probs = np.sum(all_test_probs, axis=1)
# argmax gets the integer index (0-4), class_labels maps it back to the string name
final_sum_predictions = class_labels[np.argmax(sum_test_probs, axis=1)]
sum_accuracy = accuracy_score(y_test, final_sum_predictions)

# 4. DISPLAY RESULTS
print("=== FUSION RULE RESULTS ===")
print(f"Product Rule Accuracy: {prod_accuracy * 100:.2f}%")
print(f"Sum Rule Accuracy:     {sum_accuracy * 100:.2f}%")

=== FUSION RULE RESULTS ===
Product Rule Accuracy: 61.36%
Sum Rule Accuracy:     61.36%


In [12]:
import numpy as np
from sklearn.metrics import accuracy_score

print("=== ACCURACY BY FREQUENCY ZONE ===")
print(f"{'Zone':<6} | {'Train Accuracy':<15} | {'Test Accuracy':<15}")
print("-" * 42)

zone_train_accs = []
zone_test_accs = []

for i in range(10):
    # 1. Slice the features for the current zone
    X_zone_train = X_train_3d[:, i, :]
    X_zone_test = X_test_3d[:, i, :]
    
    # 2. Get standard predictions (not probabilities) from the i-th SVM
    train_preds = models[i].predict(X_zone_train)
    test_preds = models[i].predict(X_zone_test)
    
    # 3. Calculate accuracy
    train_acc = accuracy_score(y_train, train_preds)
    test_acc = accuracy_score(y_test, test_preds)
    
    zone_train_accs.append(train_acc)
    zone_test_accs.append(test_acc)
    
    # 4. Print the results in a formatted table
    print(f"Zone {i+1:<2} | {train_acc * 100:>13.2f}% | {test_acc * 100:>13.2f}%")

print("-" * 42)
print(f"{'Mean':<6} | {np.mean(zone_train_accs) * 100:>13.2f}% | {np.mean(zone_test_accs) * 100:>13.2f}%")

=== ACCURACY BY FREQUENCY ZONE ===
Zone   | Train Accuracy  | Test Accuracy  
------------------------------------------
Zone 1  |         64.57% |         43.18%
Zone 2  |         57.14% |         52.27%
Zone 3  |         57.14% |         50.00%
Zone 4  |         57.14% |         43.18%
Zone 5  |         52.00% |         50.00%
Zone 6  |         58.86% |         54.55%
Zone 7  |         49.14% |         43.18%
Zone 8  |         57.71% |         54.55%
Zone 9  |         52.57% |         43.18%
Zone 10 |         49.71% |         52.27%
------------------------------------------
Mean   |         55.60% |         48.64%


In [13]:
# Change this:
# all_test_probs[:, i, :] = models[i].predict_proba(X_zone_test)

# To this:
all_test_probs[:, i, :] = optimized_models[i].predict_proba(X_zone_test)

In [ ]:
'''With GridSearchCV'''
# Create an array to hold the probabilities
num_languages = len(np.unique(y_train)) 
all_test_probs = np.zeros((len(test_set), 10, num_languages))

# 1. Gather probability predictions from all 10 models
for i in range(10):
    X_zone_test = X_test_3d[:, i, :]
    # all_test_probs[:, i, :] = models[i].predict_proba(X_zone_test)
    all_test_probs[:, i, :] = optimized_models[i].predict_proba(X_zone_test)

# Fetch the exact string labels the SVM learned (e.g., ['FR', 'IT', 'PO', 'SA', 'SI'])
class_labels = models[0].classes_

# 2. PRODUCT RULE FUSION
prod_test_probs = np.prod(all_test_probs, axis=1)
# argmax gets the integer index (0-4), class_labels maps it back to the string name
final_prod_predictions = class_labels[np.argmax(prod_test_probs, axis=1)] 
prod_accuracy = accuracy_score(y_test, final_prod_predictions)

# 3. SUM RULE FUSION
sum_test_probs = np.sum(all_test_probs, axis=1)
# argmax gets the integer index (0-4), class_labels maps it back to the string name
final_sum_predictions = class_labels[np.argmax(sum_test_probs, axis=1)]
sum_accuracy = accuracy_score(y_test, final_sum_predictions)

# 4. DISPLAY RESULTS
print("=== FUSION RULE RESULTS ===")
print(f"Product Rule Accuracy: {prod_accuracy * 100:.2f}%")
print(f"Sum Rule Accuracy:     {sum_accuracy * 100:.2f}%")

=== FUSION RULE RESULTS ===
Product Rule Accuracy: 63.64%
Sum Rule Accuracy:     63.64%


In [15]:
import numpy as np
from sklearn.metrics import accuracy_score

print("=== TUNED ACCURACY BY FREQUENCY ZONE ===")
print(f"{'Zone':<6} | {'Train Accuracy':<15} | {'Test Accuracy':<15}")
print("-" * 42)

zone_train_accs = []
zone_test_accs = []

for i in range(10):
    # 1. Slice the features for the current zone
    X_zone_train = X_train_3d[:, i, :]
    X_zone_test = X_test_3d[:, i, :]
    
    # 2. Get standard predictions from the GridSearchCV optimized models
    train_preds = optimized_models[i].predict(X_zone_train)
    test_preds = optimized_models[i].predict(X_zone_test)
    
    # 3. Calculate accuracy
    train_acc = accuracy_score(y_train, train_preds)
    test_acc = accuracy_score(y_test, test_preds)
    
    zone_train_accs.append(train_acc)
    zone_test_accs.append(test_acc)
    
    # 4. Print the results in a formatted table
    print(f"Zone {i+1:<2} | {train_acc * 100:>13.2f}% | {test_acc * 100:>13.2f}%")

print("-" * 42)
print(f"{'Mean':<6} | {np.mean(zone_train_accs) * 100:>13.2f}% | {np.mean(zone_test_accs) * 100:>13.2f}%")

=== TUNED ACCURACY BY FREQUENCY ZONE ===
Zone   | Train Accuracy  | Test Accuracy  
------------------------------------------
Zone 1  |         79.43% |         47.73%
Zone 2  |        100.00% |         40.91%
Zone 3  |         70.86% |         56.82%
Zone 4  |         93.14% |         47.73%
Zone 5  |         76.00% |         50.00%
Zone 6  |        100.00% |         54.55%
Zone 7  |        100.00% |         52.27%
Zone 8  |         98.86% |         43.18%
Zone 9  |         62.86% |         50.00%
Zone 10 |         61.71% |         50.00%
------------------------------------------
Mean   |         84.29% |         49.32%
